# Paper Financial Comparison Plots

This notebook creates separate publication-quality financial figures from saved `daily_financial_detail.csv` files. Optimization notebooks only save CSV outputs; all paper figures are centralized here.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import ticker
from matplotlib.patches import Patch

PROJECT = Path.cwd()
OUT_DIR = PROJECT / "Paper_Figures" / "financial_comparison"
OUT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 9.5,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "figure.dpi": 160,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

SCENARIOS = [
    ("Shrinking", "Perfect", "Shrinking-Perfect", PROJECT / "Results/Plots/Cost/2025_PerfectSessionkWh_PerfectNumbEV_PerfectatArrival/daily_financial_detail.csv"),
    ("Shrinking", "Persistence", "Shrinking-Persistence", PROJECT / "Results/Plots/Cost/2025_PersistenceSessionkWh_PersistenceNumbEV_PerfectatArrival/daily_financial_detail.csv"),
    ("Rolling", "Perfect", "Rolling-Perfect", PROJECT / "Results_Rolling/Plots/Cost/2025_PerfectSessionkWh_PerfectNumbEV_PerfectatArrival/daily_financial_detail.csv"),
    ("Rolling", "Persistence", "Rolling-Persistence", PROJECT / "Results_Rolling/Plots/Cost/2025_PersistenceSessionkWh_PersistenceNumbEV_PerfectatArrival/daily_financial_detail.csv"),
]
SCENARIO_ORDER = [s[2] for s in SCENARIOS]
CASE_ORDER = ["Retail only", "Both"]
CASE_COLORS = {"Retail only": "#D55E00", "Both": "#0072B2"}
COMPONENT_COLORS = {
    "EV Revenue": "#009E73",
    "WM Revenue": "#0072B2",
    "TOU Cost": "#CC79A7",
    "PD Cost": "#E69F00",
    "NCD Cost": "#6A3D9A",
}
METRICS = ["Total Revenue", "WM Revenue", "TOU Cost", "PD Cost", "NCD Cost", "EV Revenue"]

def dollars(x, pos=None):
    sign = "-" if x < 0 else ""
    x = abs(x)
    if x >= 1000:
        return f"{sign}${x/1000:.0f}k"
    return f"{sign}${x:.0f}"

def save_fig(fig, filename):
    path = OUT_DIR / filename
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    print(path.relative_to(PROJECT))
    return path


In [ ]:
frames = []
missing = []
for mpc, forecast, scenario, path in SCENARIOS:
    if not path.exists():
        missing.append(path.relative_to(PROJECT))
        continue
    df = pd.read_csv(path)
    df["Date"] = pd.to_datetime(df["Date"])
    df["MPC"] = mpc
    df["Forecast"] = forecast
    df["Scenario"] = scenario
    frames.append(df)

if missing:
    print("Missing input files:")
    for item in missing:
        print(" -", item)
if not frames:
    raise FileNotFoundError("No daily_financial_detail.csv files found.")

all_df = pd.concat(frames, ignore_index=True)
all_df = all_df[all_df["Case"].isin(CASE_ORDER)].copy()
for col in METRICS + ["WM BESS Total", "WM EV Total", "WM Energy Revenue", "WM Capacity Revenue"]:
    if col in all_df.columns:
        all_df[col] = pd.to_numeric(all_df[col], errors="coerce")

summary = all_df.groupby(["Scenario", "MPC", "Forecast", "Case"], as_index=False)[METRICS].sum(numeric_only=True)
summary.to_csv(OUT_DIR / "financial_summary_by_scenario_case.csv", index=False)

wide = all_df.pivot_table(index=["Scenario", "MPC", "Forecast", "Date"], columns="Case", values=METRICS, aggfunc="sum")
delta_df = pd.DataFrame(index=wide.index).reset_index()
for metric in METRICS:
    if (metric, "Both") in wide.columns and (metric, "Retail only") in wide.columns:
        delta_df[metric] = (wide[(metric, "Both")] - wide[(metric, "Retail only")]).values

delta_df.to_csv(OUT_DIR / "financial_delta_both_minus_retail_daily.csv", index=False)
print(f"Loaded {len(all_df)} case-day rows from {len(frames)} scenario files.")
print(f"Date range: {all_df['Date'].min().date()} to {all_df['Date'].max().date()}")
summary

## Figure 1: Daily Total Value Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7.6, 4.2), constrained_layout=True)
positions, data, colors, labels = [], [], [], []
x = 1.0
for scenario in SCENARIO_ORDER:
    if scenario not in set(all_df["Scenario"]):
        continue
    for case in CASE_ORDER:
        vals = all_df[(all_df["Scenario"] == scenario) & (all_df["Case"] == case)]["Total Revenue"].dropna().values
        if len(vals) == 0:
            continue
        positions.append(x)
        data.append(vals)
        colors.append(CASE_COLORS[case])
        labels.append(scenario.replace("-", "\n") + "\n" + case)
        x += 0.82
    x += 0.45

parts = ax.violinplot(data, positions=positions, widths=0.58, showmedians=True, showextrema=False)
for body, color in zip(parts["bodies"], colors):
    body.set_facecolor(color)
    body.set_edgecolor("#222222")
    body.set_alpha(0.28)
    body.set_linewidth(0.8)
parts["cmedians"].set_color("#222222")
parts["cmedians"].set_linewidth(1.1)

rng = np.random.default_rng(7)
for pos, vals, color in zip(positions, data, colors):
    jitter = rng.normal(0, 0.055, size=len(vals))
    ax.scatter(np.full(len(vals), pos) + jitter, vals, s=13, color=color, edgecolor="white", linewidth=0.35, alpha=0.78, zorder=3)
ax.axhline(0, color="#333333", lw=0.8)
ax.set_xticks(positions)
ax.set_xticklabels(labels, rotation=35, ha="right")
ax.set_ylabel("Daily net financial value (US$)")
ax.yaxis.set_major_formatter(ticker.FuncFormatter(dollars))
ax.grid(axis="y", color="#bdbdbd", alpha=0.32, linewidth=0.7)
ax.legend(handles=[Patch(facecolor=CASE_COLORS[c], edgecolor="none", label=c, alpha=0.75) for c in CASE_ORDER], frameon=False, loc="upper left")
ax.set_title("Daily financial value distribution across MPC and forecast scenarios", fontweight="bold")
save_fig(fig, "fig_financial_violin_daily_total.png")
plt.show()

## Figure 2: Wholesale Participation Delta Heatmap

In [ ]:
heat_metrics = ["Total Revenue", "WM Revenue", "TOU Cost", "PD Cost", "NCD Cost"]
heat = delta_df.groupby("Scenario")[heat_metrics].sum(numeric_only=True).reindex(SCENARIO_ORDER)
fig, ax = plt.subplots(figsize=(7.0, 3.35), constrained_layout=True)
max_abs = float(np.nanmax(np.abs(heat.values))) if heat.size else 1.0
im = ax.imshow(heat.values, cmap="RdBu", vmin=-max_abs, vmax=max_abs, aspect="auto")
ax.set_xticks(np.arange(len(heat_metrics)))
ax.set_xticklabels([m.replace(" Revenue", "\nRevenue").replace(" Cost", "\nCost") for m in heat_metrics])
ax.set_yticks(np.arange(len(heat.index)))
ax.set_yticklabels([s.replace("-", "\n") for s in heat.index])
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        val = heat.iloc[i, j]
        text = f"{val/1000:.1f}k" if abs(val) >= 1000 else f"{val:.0f}"
        ax.text(j, i, text, ha="center", va="center", fontsize=8, color="#111111")
ax.set_title("Incremental value of wholesale-enabled operation (Both minus retail-only)", fontweight="bold")
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.025)
cbar.ax.yaxis.set_major_formatter(ticker.FuncFormatter(dollars))
cbar.set_label("US$ over analyzed period")
save_fig(fig, "fig_financial_delta_heatmap.png")
plt.show()

## Figure 3: Value Stack for Wholesale-Enabled Operation

In [ ]:
components = ["EV Revenue", "WM Revenue", "TOU Cost", "PD Cost", "NCD Cost"]
both = summary[summary["Case"] == "Both"].set_index("Scenario").reindex(SCENARIO_ORDER)
fig, ax = plt.subplots(figsize=(7.4, 4.0), constrained_layout=True)
xs = np.arange(len(both))
pos_bottom = np.zeros(len(both))
neg_bottom = np.zeros(len(both))
for comp in components:
    vals = both[comp].fillna(0).values
    bottoms = np.where(vals >= 0, pos_bottom, neg_bottom)
    ax.bar(xs, vals, bottom=bottoms, width=0.62, color=COMPONENT_COLORS[comp], edgecolor="white", linewidth=0.6, label=comp)
    pos_bottom += np.where(vals >= 0, vals, 0)
    neg_bottom += np.where(vals < 0, vals, 0)
net = both["Total Revenue"].fillna(0).values
ax.plot(xs, net, color="#111111", marker="o", markersize=4.5, linewidth=1.35, label="Net total")
for xi, yi in zip(xs, net):
    ax.annotate(f"${yi:,.0f}", (xi, yi), textcoords="offset points", xytext=(0, 7 if yi >= 0 else -13), ha="center", fontsize=8)
ax.axhline(0, color="#333333", lw=0.8)
ax.set_xticks(xs)
ax.set_xticklabels([s.replace("-", "\n") for s in both.index])
ax.set_ylabel("Run-period value (US$)")
ax.yaxis.set_major_formatter(ticker.FuncFormatter(dollars))
ax.grid(axis="y", color="#bdbdbd", alpha=0.3, linewidth=0.7)
ax.legend(ncol=3, frameon=False, loc="upper left", bbox_to_anchor=(0.0, 1.02))
ax.set_title("Value-stack decomposition for wholesale-enabled operation", fontweight="bold")
save_fig(fig, "fig_financial_value_stack_both.png")
plt.show()

## Figure 4: Daily Wholesale Delta Distribution

In [ ]:
metric = "Total Revenue"
fig, ax = plt.subplots(figsize=(6.6, 3.7), constrained_layout=True)
positions, data = [], []
for i, scenario in enumerate(SCENARIO_ORDER, 1):
    vals = delta_df[delta_df["Scenario"] == scenario][metric].dropna().values
    if len(vals):
        positions.append(i)
        data.append(vals)
parts = ax.boxplot(data, positions=positions, widths=0.5, patch_artist=True, showfliers=False, medianprops={"color":"#111111", "linewidth":1.2})
for patch in parts["boxes"]:
    patch.set_facecolor("#9ecae1")
    patch.set_edgecolor("#22577a")
    patch.set_alpha(0.65)
for pos, vals in zip(positions, data):
    ax.scatter(np.full(len(vals), pos), vals, s=14, color="#22577a", alpha=0.55, edgecolor="white", linewidth=0.3)
ax.axhline(0, color="#333333", lw=0.8)
ax.set_xticks(positions)
ax.set_xticklabels([SCENARIO_ORDER[p-1].replace("-", "\n") for p in positions])
ax.set_ylabel("Daily total value delta (US$)\nBoth minus retail-only")
ax.yaxis.set_major_formatter(ticker.FuncFormatter(dollars))
ax.grid(axis="y", color="#bdbdbd", alpha=0.3, linewidth=0.7)
ax.set_title("Daily financial impact of wholesale participation", fontweight="bold")
save_fig(fig, "fig_financial_daily_delta_box.png")
plt.show()

## Figure 5: Retail-Only and Wholesale-Enabled Daily Comparison

In [ ]:
# Figure 5: Daily Retail-only vs Both comparison in aligned subplots.
fig, axes = plt.subplots(2, 2, figsize=(10.2, 5.8), sharex=False, sharey=True, constrained_layout=True)
axes = axes.flatten()
bar_w = 0.38
for ax, scenario in zip(axes, SCENARIO_ORDER):
    sub = all_df[all_df["Scenario"] == scenario].copy()
    if sub.empty:
        ax.set_visible(False)
        continue
    dates = sorted(sub["Date"].unique())
    x = np.arange(len(dates))
    retail = sub[sub["Case"] == "Retail only"].set_index("Date").reindex(dates)["Total Revenue"].values
    both = sub[sub["Case"] == "Both"].set_index("Date").reindex(dates)["Total Revenue"].values
    delta = both - retail
    ax.bar(x - bar_w/2, retail, width=bar_w, color=CASE_COLORS["Retail only"], alpha=0.78, label="Retail only")
    ax.bar(x + bar_w/2, both, width=bar_w, color=CASE_COLORS["Both"], alpha=0.78, label="Both")
    ax.plot(x, delta, color="#111111", marker="o", markersize=3.2, linewidth=1.0, label="Both - Retail")
    ax.axhline(0, color="#333333", linewidth=0.8)
    ax.set_title(scenario.replace("-", " + "), fontweight="bold", pad=5)
    ax.set_xticks(x)
    ax.set_xticklabels([pd.Timestamp(d).strftime("%m/%d") for d in dates], rotation=0)
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(dollars))
    ax.grid(axis="y", color="#bdbdbd", alpha=0.30, linewidth=0.7)
    # Label only the two largest negative deltas to avoid clutter.
    if len(delta):
        worst_idx = np.argsort(delta)[:min(2, len(delta))]
        for idx in worst_idx:
            if delta[idx] < 0:
                ax.annotate(f"{delta[idx]:.0f}", (x[idx], delta[idx]), textcoords="offset points", xytext=(0, -12), ha="center", fontsize=7.5, color="#111111")
axes[0].set_ylabel("Daily value / delta (US$)")
axes[2].set_ylabel("Daily value / delta (US$)")
handles = [
    Patch(facecolor=CASE_COLORS["Retail only"], label="Retail only", alpha=0.78),
    Patch(facecolor=CASE_COLORS["Both"], label="Both", alpha=0.78),
    plt.Line2D([0], [0], color="#111111", marker="o", linewidth=1.0, label="Both - Retail"),
]
fig.legend(handles=handles, ncol=3, frameon=False, loc="upper center", bbox_to_anchor=(0.5, 1.03))
fig.suptitle("Daily comparison of retail-only and wholesale-enabled operation", fontsize=12, fontweight="bold", y=1.08)
save_fig(fig, "fig_financial_case_comparison_subplots.png")
plt.show()

## Figure 6: Retail-Only and Wholesale-Enabled Value-Stack Subplots

In [ ]:
# Figure 6: Run-period value stack for Retail-only and Both in each scenario.
components = ["EV Revenue", "WM Revenue", "TOU Cost", "PD Cost", "NCD Cost"]
fig, axes = plt.subplots(2, 2, figsize=(10.2, 5.8), sharey=True, constrained_layout=True)
axes = axes.flatten()
for ax, scenario in zip(axes, SCENARIO_ORDER):
    scen_sum = summary[summary["Scenario"] == scenario].set_index("Case").reindex(CASE_ORDER)
    xs = np.arange(len(CASE_ORDER))
    pos_bottom = np.zeros(len(CASE_ORDER))
    neg_bottom = np.zeros(len(CASE_ORDER))
    for comp in components:
        vals = scen_sum[comp].fillna(0).values
        bottoms = np.where(vals >= 0, pos_bottom, neg_bottom)
        ax.bar(xs, vals, bottom=bottoms, width=0.58, color=COMPONENT_COLORS[comp], edgecolor="white", linewidth=0.55, label=comp)
        pos_bottom += np.where(vals >= 0, vals, 0)
        neg_bottom += np.where(vals < 0, vals, 0)
    net = scen_sum["Total Revenue"].fillna(0).values
    ax.plot(xs, net, color="#111111", marker="o", markersize=4.0, linewidth=1.1)
    for xi, yi in zip(xs, net):
        ax.annotate(f"${yi:,.0f}", (xi, yi), textcoords="offset points", xytext=(0, 6 if yi >= 0 else -12), ha="center", fontsize=7.5)
    ax.axhline(0, color="#333333", linewidth=0.8)
    ax.set_title(scenario.replace("-", " + "), fontweight="bold", pad=5)
    ax.set_xticks(xs)
    ax.set_xticklabels(CASE_ORDER)
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(dollars))
    ax.grid(axis="y", color="#bdbdbd", alpha=0.30, linewidth=0.7)
axes[0].set_ylabel("Run-period value (US$)")
axes[2].set_ylabel("Run-period value (US$)")
handles = [Patch(facecolor=COMPONENT_COLORS[c], edgecolor="none", label=c) for c in components]
fig.legend(handles=handles, ncol=5, frameon=False, loc="upper center", bbox_to_anchor=(0.5, 1.03))
fig.suptitle("Retail-only and wholesale-enabled value stacks by scenario", fontsize=12, fontweight="bold", y=1.08)
save_fig(fig, "fig_financial_case_value_stack_subplots.png")
plt.show()